# Prompt Chaining II

# Task
รับคำถามจากผู้ใช้งาน ซึ่งเป็นคำถามที่ต้องการค้นหาข้อมูลจากฐานข้อมูล (Database) จากนั้นให้ AI Agent ทำงานตามขั้นตอนเพื่อค้นหาคำตอบหรือสร้างกราฟที่เหมาะสม

การทำงานของ Agent:
Agent จะต้องแยกประเภทคำถาม (Classify) ออกเป็น 2 ประเภท และทำงานตาม Flow ดังต่อไปนี้:

## 1. Text Question
ลักษณะคำถาม: เป็นคำถามตรงไปตรงมาที่ผู้ใช้คาดหวังคำตอบสั้นๆ เช่น ตัวเลข 1-2 จำนวน หรือประโยคคำตอบสั้นๆ

#### กระบวนการทำงาน:

1. แปลงคำถามของผู้ใช้ให้เป็นคำสั่ง SQL โดยอ้างอิงจากโครงสร้างฐานข้อมูล (DB Schema)

2. ตรวจสอบความถูกต้องของ SQL (SQL Validation): ตรวจสอบว่าคำสั่ง SQL ถูกต้องหรือไม่ หากไม่ถูกต้อง ระบบจะส่งฟีดแบ็ก (Feedback) กลับไปให้แก้ไขและสร้าง SQL ใหม่จนกว่าจะถูกต้อง

3. รันคำสั่ง SQL (Execute) เพื่อดึงข้อมูลจากฐานข้อมูล

4. นำผลลัพธ์ที่ได้มาเรียบเรียงเป็นคำตอบภาษาธรรมชาติ (Natural language) ที่อ่านเข้าใจง่าย

**ตัวอย่างคำถาม**: "ใครมีจำนวนอัลบั้มมากที่สุดในปี 2020 ?" (Who has the most number of album in 2020?)

##  2. Plot Question 
ลักษณะคำถาม: เป็นคำถามที่ผู้ใช้ต้องการเห็นภาพรวม การเปรียบเทียบ แนวโน้ม หรือการกระจายตัวของข้อมูล ซึ่งต้องอาศัยการแสดงผลแบบกราฟ (Visualization) เพื่อให้เข้าใจได้ง่ายขึ้น

#### กระบวนการทำงาน:

1. แปลงคำถามให้เป็นคำสั่ง SQL ที่สามารถดึงข้อมูลออกมาเป็นชุด (Multi-row data / Time-series) ที่เหมาะสำหรับการสร้างกราฟ

2. ตรวจสอบความถูกต้องของ SQL (SQL Validation): เช่นเดียวกับแบบแรก หากผิดพลาดให้วนลูปกลับไปแก้ไข

3. รันคำสั่ง SQL (Execute) เพื่อดึงข้อมูลจากฐานข้อมูล

4. นำข้อมูลผลลัพธ์ที่ได้มาสร้างเป็นโค้ดกราฟในรูปแบบ Mermaid.js

5. ตรวจสอบความถูกต้องของกราฟ (Mermaid Validation): ให้ LLM ตรวจสอบว่าโค้ด Mermaid ถูกต้องตามหลักไวยากรณ์ (Syntax) หรือไม่ และเช็คว่าข้อมูลในกราฟสามารถตอบคำถามของผู้ใช้ได้จริงหรือไม่

    - หากถูกต้อง: แสดงผลกราฟให้ผู้ใช้

    - หากไม่ถูกต้อง: (เช่น ข้อมูลไม่พอสร้างกราฟ, โค้ดผิด) ระบบจะนำฟีดแบ็กที่ได้ วนลูปกลับไปเริ่มต้นสร้างคำสั่ง SQL ใหม่ เพื่อดึงข้อมูลให้ถูกต้องตั้งแต่ต้น

**ตัวอย่างคำถาม**: "ใครมีจำนวนอัลบั้มมากที่สุดในแต่ละปี และมีจำนวนเท่าไร ?" (Who has the most number of album each year and by how many? plot a graph)


## STEP1: Design

In [1]:
from IPython.display import Image
from IPython.core.display import HTML 
Image(url= "./Examples/prompt-chain-iii.png", width=700)

In [2]:
# !uv pip install langchain-ollama langgraph pydantic

In [12]:
# !uv pip install mermaid-py
# 

## STEP2: Build Graph

In [13]:
from typing import Annotated, TypedDict, Optional, Literal, Tuple
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

In [14]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    query_type: Optional[str]
    
    # SQL Generation & Validation State
    sql_query: Optional[str]
    sql_feedback: Optional[str]
    is_sql_valid: Optional[bool]
    
    # Execution & Result State
    query_result: Optional[str]
    
    # Mermaid Generation & Validation State
    mermaid_code: Optional[str]
    mermaid_feedback: Optional[str]
    is_mermaid_valid: Optional[bool]

class QueryClassification(BaseModel):
    query_type: Literal["text question", "plot question"] = Field(...)

class SQLQuery(BaseModel):
    sql_query: str = Field(description="The valid SQL query.")

class MermaidValidationResult(BaseModel):
    is_valid: bool = Field(description="True if the mermaid code is syntactically correct and accurately answers the user's question based on the data. False otherwise.")
    feedback: str = Field(description="If invalid, provide specific feedback on what is wrong. If valid, return 'Valid'.")

class MermaidOutput(BaseModel):
    mermaid_code: str = Field(description="The raw mermaid.js code.")
    

In [15]:
llm = ChatOllama(model="scb10x/typhoon2.5-qwen3-4b")
classifier_llm = llm.with_structured_output(QueryClassification)
sql_llm = llm.with_structured_output(SQLQuery)
mermaid_llm = llm.with_structured_output(MermaidOutput)
mermaid_validator_llm = llm.with_structured_output(MermaidValidationResult)

### External Tools

In [16]:
import mermaid as md
from mermaid.graph import Graph

def render_mermaid(code):
    return md.Mermaid(code)
    
# render_mermaid("""
# stateDiagram-v2
#     [*] --> Still
#     Still --> [*]

#     Still --> Moving
#     Moving --> Still
#     Moving --> Crash
#     Crash --> [*]
# """)

In [17]:
sqlSchema = '''
CREATE TABLE "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION
);

CREATE TABLE "genres"
(
    [GenreId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "media_types"
(
    [MediaTypeId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "tracks"
(
    [TrackId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(200)  NOT NULL,
    [AlbumId] INTEGER,
    [MediaTypeId] INTEGER  NOT NULL,
    [GenreId] INTEGER,
    [Composer] NVARCHAR(220),
    [Milliseconds] INTEGER  NOT NULL,
    [Bytes] INTEGER,
    [UnitPrice] NUMERIC(10,2)  NOT NULL,
    FOREIGN KEY ([AlbumId]) REFERENCES "albums" ([AlbumId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION,
    FOREIGN KEY ([GenreId]) REFERENCES "genres" ([GenreId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION,
    FOREIGN KEY ([MediaTypeId]) REFERENCES "media_types" ([MediaTypeId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION
);
'''

In [18]:
import sqlite3
import os

def execute_query(query):
    """
    Connects to a SQLite database, executes a given query, and returns the results.

    Args:
        query (str): The SQL query string to execute.

    Returns:
        list: A list of tuples, where each tuple represents a row from the results.
              Returns None if an error occurs.
    """
    connection = None
    results = None
    
    try:
        db_file = "./Examples/example.db"
        connection = sqlite3.connect(db_file)
        cursor = connection.cursor()
        
        print(f"Executing query:\n{query}\n")
        cursor.execute(query)
        results = cursor.fetchall()
        
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        
    finally:
        if connection:
            connection.close()
            
    return results

In [19]:
execute_query("SELECT * FROM albums LIMIT 5")

Executing query:
SELECT * FROM albums LIMIT 5



[(1, 'For Those About To Rock We Salute You', 1),
 (2, 'Balls to the Wall', 2),
 (3, 'Restless and Wild', 2),
 (4, 'Let There Be Rock', 1),
 (5, 'Big Ones', 3)]

In [20]:
def sql_validation(query: str) -> bool:
    """
    Validates if the given SQL query is syntactically correct without actually fetching or modifying data.
    
    Args:
        query (str): The SQL query string to validate.
    """
    try:
        db_file = "./Examples/example.db"
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        cursor.execute(f"EXPLAIN {query}")
        conn.close()
        return True, None
    except Exception as e:
        return False, str(e)

In [21]:
sql_validation("SELECT * FROM albums LIMIT 5")

(True, None)

In [22]:
sql_validation("SELECT * FROM albumsx LIMIT 5")

(False, 'no such table: albumsx')

### Define Nodes

In [23]:
def ClassifierNode(state: AgentState):
    messages = state["messages"]
    user_question = [m for m in messages if isinstance(m, HumanMessage)][-1].content
    
    sys_prompt = """You are an expert AI router. Classify the user's question into one of two categories:
    1. 'text question': The user is asking a direct question that expects a concise answer, such as 1-2 numbers or a short, simple sentence. (e.g., 'Who had the most albums in 2020?', 'What is the total revenue?').
    2. 'plot question': The user needs a visual representation to understand the data properly. This involves trends, distributions, or comparisons across multiple data points. (e.g., 'Who has the most albums each year and by how many? plot a graph', 'Show me sales over the last 5 months.')."""

    result = classifier_llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=user_question)])
    print(f"--- [Classifier] Classified as: {result.query_type} ---")
    return {"query_type": result.query_type}
    # return {"query_type": "plot question"}

In [24]:
def TextSQLGeneratorNode(state: AgentState):
    messages = state["messages"]
    user_question = [m for m in messages if isinstance(m, HumanMessage)][-1].content
    feedback = state.get("sql_feedback", None)
    prev_sql_query = state.get("sql_query", None)

    
    sys_prompt = ""
    sys_prompt += "You are an expert SQL Data Analyst. Generate a SQL query based on this schema:\n\n"
    sys_prompt += f"{sqlSchema}\n\n"
    sys_prompt += f"Make sure the SQL query is suitable for a simple answer (1-2 numbers or a short sentence)\n\n"

    if prev_sql_query is not None and prev_sql_query!="":
        sys_prompt += f"Here is your previous attempt on the SQL query: {prev_sql_query}\n\n"
        
    if feedback is not None and feedback!="":
        sys_prompt += f"You have failed previously, here is the SQL Error Feedback: {feedback}\n"
        sys_prompt += f"Fix the errors and generate a valid SQL query."

    print(f"--- Generating SQL ... ---")
    print(f"--- [Prompt]: {sys_prompt} ---")
    print()
    
    result = sql_llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=user_question)])
    print(f"--- Generated SQL: {result.sql_query} ---")
    return {"sql_query": result.sql_query}

In [25]:
def PlotSQLGeneratorNode(state: AgentState):
    messages = state["messages"]
    user_question = [m for m in messages if isinstance(m, HumanMessage)][-1].content
    
    prev_sql_query = state.get("sql_query", None)
    prev_mermaid_code = state.get("mermaid_code", None)
    
    sql_feedback = state.get("sql_feedback", "")
    mermaid_feedback = state.get("mermaid_feedback", "")


    sys_prompt = ""
    sys_prompt += "You are an expert SQL Data Analyst. Generate a SQL query that returns multi-row, structured data suitable for charting (e.g., groupings, time-series)."
    sys_prompt += f"The SQL query must be based on based on this schema\n\n"
    sys_prompt += f"{sqlSchema}\n\n"
    
    if prev_sql_query is not None and prev_sql_query!="":
        sys_prompt += f"Here is your previous attempt on the SQL query: {prev_sql_query}\n\n"
        
    if sql_feedback is not None and sql_feedback!="":
        sys_prompt += f"You have failed previously, here is the SQL Error Feedback: {sql_feedback}\n"
        sys_prompt += f"Fix the errors and generate a valid SQL query.\n\n"


    if prev_mermaid_code is not None and prev_mermaid_code!="":
        sys_prompt += f"Here is your previous attempt on the mermaid code: {prev_mermaid_code}\n\n"
        
    if mermaid_feedback is not None and mermaid_feedback!="":
        sys_prompt += f"You have failed previously, here is the Mermaid Plotting Feedback: {mermaid_feedback}\n"
        sys_prompt += f"Adjust SQL to return data better suited for this plot.\n\n"

    print(f"--- Generating SQL ... ---")
    print(f"--- [Prompt]: {sys_prompt} ---")
    print()
    result = sql_llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=user_question)])
    print(f"--- Generated SQL: {result.sql_query} ---")
    return {"sql_query": result.sql_query}

In [26]:
def SQLValidatorNode(state: AgentState):
    sql_query = state.get("sql_query", "")
    is_valid, feedback = sql_validation(sql_query)
    print(f"--- [SQL Validator] Valid: {is_valid} ---")
    print(f"--- [SQL Validator] Feedback: {feedback} ---")
    return {"is_sql_valid": is_valid, "sql_feedback": feedback if not is_valid else None}

def SQLExecutorNode(state: AgentState):
    # Mock DB execution
    sql_query = state.get("sql_query")
    print(f"--- Executing... ---")
    query_result = execute_query(sql_query)
    if len(query_result) > 10:
        print(f"--- [SQL Result]: {query_result[0:10]} and more... ---")
    else:
        print(f"--- [SQL Result]: {query_result} ---")
    return {"query_result": json.dumps(query_result[0:20])}

def ResponseGeneratorNode(state: AgentState):
    messages = state["messages"]
    user_question = [m for m in messages if isinstance(m, HumanMessage)][-1].content
    
    sql_query = state.get("sql_query")
    query_result = state.get("query_result")

    sys_prompt = ""
    sys_prompt += f"You are a helpful assistance\n"
    sys_prompt += f"You have excecuted this query: {sql_query}\n\n"
    sys_prompt += f"Here is the query's result: {query_result}\n\n"
    sys_prompt += f"Base your answer SOLELY on this DB result. Answer the following question.\n"

    print(f"--- Generating Response... ---")
    response = llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=user_question)])
    return {"messages": [response]}

In [27]:
import json

def MermaidGeneratorNode(state: AgentState):
    messages = state["messages"]
    user_question = [m for m in messages if isinstance(m, HumanMessage)][-1].content
    
    sql_query = state.get("sql_query")
    query_result = state.get("query_result")

    sys_prompt = ""
    sys_prompt += f"You are a data visualization expert.\n"
    sys_prompt += f"You have excecuted this query for the visualization: {sql_query}\n\n"
    sys_prompt += f"Here is the query's result: {query_result}\n\n"
    sys_prompt += f"Generate a valid Mermaid.js code from the given results to answer the user's question.\n"

    print(f"--- Generating Mermaid ... ---")
    print(f"--- [Prompt]: {sys_prompt} ---")
    print()
    
    result = mermaid_llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=user_question)])
    print(f"--- Generated Mermaid ---")
    return {"mermaid_code": result.mermaid_code}

def MermaidValidatorNode(state: AgentState):
    messages = state["messages"]
    user_question = [m for m in messages if isinstance(m, HumanMessage)][-1].content
    mermaid_code = state.get("mermaid_code", "")
    query_result = state.get("query_result", "")
    

    sys_prompt = ""
    sys_prompt += f"You are an expert Data Visualizer and Mermaid.js validator.\n"
    sys_prompt += f"User's Question: {user_question}\n\n"
    sys_prompt += f"Database Result: {query_result}\n\n"
    sys_prompt += f"Generated Mermaid Code:\n"
    sys_prompt += f"```mermaid\n{mermaid_code}\n```\n\n\n"
    sys_prompt += f"""Your Task:
    1. Verify that the Mermaid code is syntactically correct (e.g., correct chart type like 'pie' or 'xychart-beta', proper data formatting).
    2. Verify that the chart semantically answers the user's question using the provided Database Result.

    If it is perfect, set `is_valid` to True and `feedback` to 'Valid'.
    If there are syntax errors, or if the SQL query did not return the right shape of data for this chart, set `is_valid` to False. In your `feedback`, explain EXACTLY what went wrong so the SQL Generator or Plot Generator can fix it in the next attempt.
    """

    print(f"--- Validating Mermaid ... ---")
    print(f"--- [Prompt]: {sys_prompt} ---")
    print()
    
    result = mermaid_validator_llm.invoke([SystemMessage(content=sys_prompt)])
    print(f"--- [Mermaid Validator] Valid: {result.is_valid} ---")
    print(f"--- [Mermaid Validator] Feedback: {result.feedback} ---")
    
    if result.is_valid:
        # final_msg = AIMessage(content=f"Here is your plot:\n```mermaid\n{mermaid_code}\n```")
        return {
            "is_mermaid_valid": result.is_valid, 
            "mermaid_feedback": result.feedback, 
            # "messages": [final_msg]
        }
    else:
        return {
            "is_mermaid_valid": result.is_valid, 
            "mermaid_feedback": result.feedback
        }

In [28]:
# xychart
#     title "Sales Revenue"
#     x-axis [jan, feb, mar, apr, may, jun, jul, aug, sep, oct, nov, dec]
#     y-axis "Revenue (in $)" 4000 --> 11000
#     bar [5000, 6000, 7500, 8200, 9500, 10500, 11000, 10200, 9200, 8500, 7000, 6000]



### Define Edges & Logic

In [29]:
def route_after_classifier(state: AgentState):
    print("Route [route_after_classifier]: ", state.get("query_type"))
    return "TextSQLGeneratorNode" if state.get("query_type") == "text question" else "PlotSQLGeneratorNode"

def route_simple_sql_validation(state: AgentState):
    print("Route [route_simple_sql_validation]: ", state.get("is_sql_valid"))
    return "TextSQLExecutorNode" if state.get("is_sql_valid") else "TextSQLGeneratorNode"

def route_plot_sql_validation(state: AgentState):
    print("Route [route_plot_sql_validation]: ", state.get("is_sql_valid"))
    return "PlotSQLExecutorNode" if state.get("is_sql_valid") else "PlotSQLGeneratorNode"

def route_mermaid_validation(state: AgentState):
    print("Route [route_mermaid_validation]: ", state.get("is_mermaid_valid"))
    return END if state.get("is_mermaid_valid") else "PlotSQLGeneratorNode"

### Define the graph

In [30]:
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("ClassifierNode", ClassifierNode)
workflow.add_node("TextSQLGeneratorNode", TextSQLGeneratorNode)
workflow.add_node("PlotSQLGeneratorNode", PlotSQLGeneratorNode)
workflow.add_node("TextSQLValidatorNode", SQLValidatorNode)
workflow.add_node("PlotSQLValidatorNode", SQLValidatorNode)
workflow.add_node("TextSQLExecutorNode", SQLExecutorNode)
workflow.add_node("PlotSQLExecutorNode", SQLExecutorNode)
workflow.add_node("ResponseGeneratorNode", ResponseGeneratorNode)
workflow.add_node("MermaidGeneratorNode", MermaidGeneratorNode)
workflow.add_node("MermaidValidatorNode", MermaidValidatorNode)

# Top level routing
workflow.add_edge(START, "ClassifierNode")
workflow.add_conditional_edges("ClassifierNode", route_after_classifier, {
    "TextSQLGeneratorNode": "TextSQLGeneratorNode",
    "PlotSQLGeneratorNode": "PlotSQLGeneratorNode"
})

# Simple Flow (With Validation Loop)
workflow.add_edge("TextSQLGeneratorNode", "TextSQLValidatorNode")
workflow.add_conditional_edges("TextSQLValidatorNode", route_simple_sql_validation, {
    "TextSQLExecutorNode": "TextSQLExecutorNode",
    "TextSQLGeneratorNode": "TextSQLGeneratorNode"
})
workflow.add_edge("TextSQLExecutorNode", "ResponseGeneratorNode")
workflow.add_edge("ResponseGeneratorNode", END)

# Plot Flow (With SQL AND Mermaid Validation Loops)
workflow.add_edge("PlotSQLGeneratorNode", "PlotSQLValidatorNode")
workflow.add_conditional_edges("PlotSQLValidatorNode", route_plot_sql_validation, {
    "PlotSQLExecutorNode": "PlotSQLExecutorNode",
    "PlotSQLGeneratorNode": "PlotSQLGeneratorNode"
})
workflow.add_edge("PlotSQLExecutorNode", "MermaidGeneratorNode")
workflow.add_edge("MermaidGeneratorNode", "MermaidValidatorNode")

# Mermaid Reflection Loop back to SQL Generation
workflow.add_conditional_edges("MermaidValidatorNode", route_mermaid_validation, {
    END: END,
    "PlotSQLGeneratorNode": "PlotSQLGeneratorNode"
})

app = workflow.compile()

In [31]:
"DONE"

'DONE'

### Try

In [32]:
from IPython.display import Image, display

user_input = {"messages": [HumanMessage(content="How many albums are in the database?")]}
result = app.invoke(user_input)
current_state = result

if current_state["query_type"]=="plot question":
    print("HERE are the plot:")
    try:
        display(render_mermaid(current_state["mermaid_code"]))
    except Exception as e:
        print("Cannot plot", str(e))
else:
    print(f"HERE are the answer: {current_state['messages'][-1].content}")


--- [Classifier] Classified as: text question ---
Route [route_after_classifier]:  text question
--- Generating SQL ... ---
--- [Prompt]: You are an expert SQL Data Analyst. Generate a SQL query based on this schema:


CREATE TABLE "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION
);

CREATE TABLE "genres"
(
    [GenreId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "media_types"
(
    [MediaTypeId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "tracks"
(
    [TrackId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(200)  NOT NULL,
    [AlbumId] INTEGER,
    [MediaTypeId] INTEGER  

In [33]:
from IPython.display import Image, display

user_input = {"messages": [HumanMessage(content="Plot number of albums by the artists")]}
result = app.invoke(user_input)
current_state = result

if current_state["query_type"]=="plot question":
    print("HERE are the plot:")
    try:
        display(render_mermaid(current_state["mermaid_code"]))
    except Exception as e:
        print("Cannot plot", str(e))
else:
    print(f"HERE are the answer: {current_state['messages'][-1].content}")


--- [Classifier] Classified as: plot question ---
Route [route_after_classifier]:  plot question
--- Generating SQL ... ---
--- [Prompt]: You are an expert SQL Data Analyst. Generate a SQL query that returns multi-row, structured data suitable for charting (e.g., groupings, time-series).The SQL query must be based on based on this schema


CREATE TABLE "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION
);

CREATE TABLE "genres"
(
    [GenreId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "media_types"
(
    [MediaTypeId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "tracks"
(
    [TrackId] INTEGER

In [168]:
# import base64
# import requests
# from IPython.display import Image, display

# def render_mermaid(graph):
#     """Renders a mermaid graph using the mermaid.ink service."""
#     graph_bytes = graph.encode("utf-8")
#     base64_bytes = base64.urlsafe_b64encode(graph_bytes)
#     base64_string = base64_bytes.decode("ascii")
    
#     url = "https://mermaid.ink/img/" + base64_string
#     return Image(requests.get(url).content)